# LFW Grad-CAM — 02. Pair-conditioned Grad-CAM

원본 query embedding과 detached 원본 gallery template의 cosine
similarity만 설명합니다. PCA/PQ code를 미분하지 않으며 압축 profile은
앞 단계에서 사례를 선택한 조건으로만 연결됩니다.

`PAIR_BUNDLE_PATH`는 case manifest 순서의 `case_id`, uint8 NHWC
`query_images`, float32 512D `gallery_templates`를 포함해야 합니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
import numpy as np

from research.embeddings import (
    create_pytorch_adapter_from_spec,
    read_model_spec,
)
from research.explainability.gradcam import PairCosineGradCAM
from research.runtime.hashing import sha256_file

MODEL_SPEC_PATH = None
CASE_MANIFEST_PATH = None
PAIR_BUNDLE_PATH = None
HEATMAP_OUTPUT_PATH = None
DEVICE = "cpu"
BATCH_SIZE = 4

In [ ]:
if EXECUTE_STAGE:
    import pandas as pd
    import torch

    required_paths = [MODEL_SPEC_PATH, CASE_MANIFEST_PATH, PAIR_BUNDLE_PATH]
    if any(path is None for path in required_paths):
        raise RuntimeError("ModelSpec, case manifest, pair bundle 경로를 지정하세요.")
    spec = read_model_spec(MODEL_SPEC_PATH, verify_checkpoint=True)
    if spec.family != MODEL_NAME:
        raise ValueError("MODEL_NAME과 ModelSpec family가 다릅니다.")
    cases = pd.read_parquet(CASE_MANIFEST_PATH)
    pair_bundle = np.load(PAIR_BUNDLE_PATH, allow_pickle=False)
    case_ids = pair_bundle["case_id"].astype(str)
    expected_case_ids = cases["case_id"].astype(str).to_numpy()
    if not np.array_equal(case_ids, expected_case_ids):
        raise ValueError("pair bundle의 case_id 순서가 case manifest와 다릅니다.")
    query_images = pair_bundle["query_images"]
    gallery_templates = pair_bundle["gallery_templates"].astype(np.float32)
    if gallery_templates.shape != (len(cases), 512):
        raise ValueError("gallery_templates는 [case_count, 512]여야 합니다.")

    adapter = create_pytorch_adapter_from_spec(spec, device=DEVICE)
    analyzer = PairCosineGradCAM(
        adapter.model,
        adapter.target_layer,
        embedding_extractor=adapter.select_embedding_tensor,
    )
    heatmap_parts = []
    score_parts = []
    for start in range(0, len(cases), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(cases))
        query_tensor = adapter.preprocess(query_images[start:stop])
        gallery_tensor = torch.from_numpy(
            gallery_templates[start:stop]
        ).to(adapter.device)
        result = analyzer.generate(
            query_tensor,
            gallery_tensor,
            batch_mode="single" if stop - start == 1 else "independent",
            target_space="origin_embedding",
        )
        heatmap_parts.append(result.heatmaps)
        score_parts.append(result.target_scores)
    heatmaps = np.concatenate(heatmap_parts, axis=0)
    target_scores = np.concatenate(score_parts, axis=0)
    if WRITE_OUTPUTS:
        if HEATMAP_OUTPUT_PATH is None:
            raise RuntimeError("HEATMAP_OUTPUT_PATH를 지정하세요.")
        destination = Path(HEATMAP_OUTPUT_PATH).resolve()
        if destination.exists():
            raise FileExistsError(f"기존 heatmap을 덮어쓸 수 없습니다: {destination}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            destination,
            case_id=case_ids,
            heatmaps=heatmaps,
            target_scores=target_scores,
            target_space=np.array(["origin_embedding"]),
            model_uid=np.array([spec.model_uid]),
            checkpoint_sha256=np.array([spec.checkpoint.sha256]),
            case_manifest_sha256=np.array([sha256_file(CASE_MANIFEST_PATH)]),
            pair_bundle_sha256=np.array([sha256_file(PAIR_BUNDLE_PATH)]),
        )
    generation_summary = {
        "case_count": int(len(cases)),
        "heatmap_shape": list(heatmaps.shape),
        "score_min": float(target_scores.min()),
        "score_max": float(target_scores.max()),
        "target_space": "origin_embedding",
    }
else:
    generation_summary = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
    }
generation_summary

heatmap은 target layer 해상도로 저장됩니다. 시각화할 때만 입력 crop
크기로 보간하며, 후속 지표 계산에는 원본 heatmap 배열과 provenance를
유지합니다.